<a href="https://colab.research.google.com/github/ayushi777lodhi-stack/MARL/blob/main/MARL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install "stable-baselines3[extra]"

In [ ]:
import os
import gymnasium as gym

In [ ]:
import random
from collections import deque
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from gymnasium import spaces
import numpy as np
import copy

In [ ]:
class MultiAgentGrid(gym.Env):

    def __init__(self):
        self.action_space=spaces.MultiDiscrete([10, 10])
        self.observation_space=spaces.Box(low=-np.inf,high=np.inf,shape=(2, 15),dtype=np.float32)
        self.agent_A_pos=None
        self.agent_B_pos=None
        self.goal_pos=np.array([4, 5])
        self.max_steps=100
        self.step_count=0
        self.agent_A_food=0
        self.agent_B_food=0
        self.agent_A_wood=0
        self.agent_B_wood=0
        self.agent_A_energy=50
        self.agent_B_energy=50
        self.max_food_spawn=10
        self.max_wood_spawn=10
        self.food_resources=[]
        self.wood_resources=[]
        self.times_closer=0
        self.times_shared=0
        self.times_traded=0
        self.times_food_collected=0
        self.times_wood_collected=0

    def are_adjacent(self, pos_A, pos_B):
        return np.linalg.norm(pos_A-pos_B)==1

    def reset(self, seed=None, options=None):

        super().reset(seed=seed)
        self.agent_A_pos=np.array([0,0])
        self.agent_B_pos=np.array([4,0])
        self.step_count=0
        self.agent_A_food=0
        self.agent_B_food=0
        self.agent_A_wood=0
        self.agent_B_wood=0
        self.agent_A_energy=50
        self.agent_B_energy=50
        self.times_closer=0
        self.times_shared=0
        self.times_traded=0
        self.times_food_collected=0
        self.times_wood_collected=0
        self.food_resources=self.spawn_food()
        self.wood_resources=self.spawn_wood()

        observation=self.observations()

        info={}

        return observation, info

    def observations(self):
        if len(self.food_resources)>0:
            food_1=self.food_resources[0]
        else:
            food_1=np.array([-1, -1])

        if len(self.food_resources)>1:
            food_2=self.food_resources[1]
        else:
            food_2=np.array([-1, -1])

        if len(self.wood_resources)>0:
            wood_1=self.wood_resources[0]
        else:
            wood_1=np.array([-1, -1])

        if len(self.wood_resources)>1:
            wood_2=self.wood_resources[1]
        else:
            wood_2=np.array([-1, -1])


        observation_A=np.array([
            self.agent_A_pos[0],
            self.agent_A_pos[1],
            self.agent_B_pos[0],
            self.agent_B_pos[1],
            food_1[0],
            food_1[1],
            food_2[0],
            food_2[1],
            wood_1[0],
            wood_1[1],
            wood_2[0],
            wood_2[1],
            self.agent_A_food,
            self.agent_A_wood,
            self.agent_A_energy/100
        ], dtype=np.float32)

        observation_B=np.array([
            self.agent_B_pos[0],
            self.agent_B_pos[1],
            self.agent_A_pos[0],
            self.agent_A_pos[1],
            food_1[0],
            food_1[1],
            food_2[0],
            food_2[1],
            wood_1[0],
            wood_1[1],
            wood_2[0],
            wood_2[1],
            self.agent_B_food,
            self.agent_B_wood,
            self.agent_B_energy/100
        ], dtype=np.float32)

        return np.array(
            [observation_A, observation_B],
            dtype=np.float32
        )

    def move(self,position,action):

        new_position=position.copy()

        if action==0:
            new_position[0]-=1

        elif action==1:
            new_position[0]+=1

        elif action==2:
            new_position[1]-=1

        elif action==3:
            new_position[1]+=1

        new_position[0]=np.clip(new_position[0],0,5)
        new_position[1]=np.clip(new_position[1],0,5)

        return new_position


    def spawn_food(self):
        food_resources=[]
        while len(food_resources)<self.max_food_spawn:
            position=np.array([
                np.random.randint(0,6),
                np.random.randint(0,6)
            ])

            if(np.array_equal(position, self.agent_A_pos) or np.array_equal(position, self.agent_B_pos)):
                continue

            duplicate=False

            for food in food_resources:
                if np.array_equal(position,food):
                    duplicate=True
                    break

            if duplicate:
                continue

            food_resources.append(position)
        return food_resources


    def spawn_wood(self):

        wood_resources=[]
        while len(wood_resources)<self.max_wood_spawn:
            position=np.array([
                np.random.randint(0,6),
                np.random.randint(0,6)
            ])


            if (np.array_equal(position, self.agent_A_pos) or np.array_equal(position, self.agent_B_pos)):
                continue

            duplicate=False

            for wood in wood_resources:
                if np.array_equal(position, wood):
                    duplicate=True
                    break

            if duplicate:
                continue

            wood_resources.append(position)

        return wood_resources


    def collect_food(self, agent_pos, agent_food, agent_id):
        if agent_id!="B":
           return agent_food, False

        for i, resource in enumerate(self.food_resources):
            if np.array_equal(agent_pos,resource):
                agent_food+=1
                self.food_resources.pop(i)
                return agent_food, True
        return agent_food, False


    def collect_wood(self, agent_pos, agent_wood, agent_id):
       if agent_id!="A":
           return agent_wood, False

       for i, resource in enumerate(self.wood_resources):
           if np.array_equal(agent_pos,resource):
               agent_wood+=1
               self.wood_resources.pop(i)
               return agent_wood, True

       return agent_wood, False


    def get_state(self):

        if len(self.food_resources)>0:
            food_1=self.food_resources[0]
        else:
            food_1=np.array([-1, -1])

        if len(self.food_resources)>1:
            food_2=self.food_resources[1]
        else:
            food_2=np.array([-1, -1])

        if len(self.wood_resources)>0:
            wood_1=self.wood_resources[0]
        else:
            wood_1=np.array([-1, -1])

        if len(self.wood_resources)>1:
            wood_2=self.wood_resources[1]
        else:
            wood_2=np.array([-1, -1])

        return np.concatenate([
            self.agent_A_pos,
            self.agent_B_pos,
            wood_1,
            wood_2,
            food_1,
            food_2,
            np.array([self.agent_A_food,
                self.agent_B_food,
                self.agent_A_wood,
                self.agent_B_wood,
                self.agent_A_energy,
                self.agent_B_energy])
            ]).astype(np.float32)


    def step(self,actions):
        reward=-2
        action_A=actions[0]
        action_B=actions[1]

        if action_A<4:
             self.agent_A_pos=self.move(self.agent_A_pos,action_A)

        if action_B<4:
            self.agent_B_pos=self.move(self.agent_B_pos,action_B)


        self.step_count+=1
        self.agent_A_energy-=2
        self.agent_B_energy-=2
        self.agent_A_energy=max(0,self.agent_A_energy)
        self.agent_B_energy=max(0,self.agent_B_energy)


        if self.are_adjacent(self.agent_A_pos,self.agent_B_pos):
            self.times_closer+=1

        both_at_goal=(np.array_equal(self.agent_A_pos,self.goal_pos) and
                      np.array_equal(self.agent_B_pos,self.goal_pos))

        if action_A==4:
            self.agent_A_food, collected=(self.collect_food(self.agent_A_pos,self.agent_A_food,"A"))

            if collected:
                reward+=5


        if action_B==4:
            self.agent_B_food,collected=(self.collect_food(self.agent_B_pos,self.agent_B_food,"B"))

            if collected:
                self.times_food_collected+=1
                reward+=5


        if action_A==5:
            if self.agent_A_food>0:
                self.agent_A_food-=1
                self.agent_A_energy=min(100,self.agent_A_energy+30)
                reward+=2


        if action_B==5:
            if self.agent_B_food>0:
                self.agent_B_food-=1
                self.agent_B_energy=min(100,self.agent_B_energy+5)
                reward+=0.5


        if action_A==6:
            if(self.agent_A_food>0 and self.are_adjacent(self.agent_A_pos,self.agent_B_pos)):
                self.agent_A_food-=1
                self.agent_B_food+=1
                self.times_shared+=1
                reward+=5


        if action_B==6:
            if(self.agent_B_food>0 and self.are_adjacent(self.agent_A_pos,self.agent_B_pos)):
                self.agent_B_food-=1
                self.agent_A_food+=1
                self.times_shared+=1
                reward+=5


        if action_A==7:
            self.agent_A_wood, collected=(self.collect_wood(self.agent_A_pos,self.agent_A_wood,"A"))

            if collected:
                self.times_wood_collected+=1
                reward+=4


        if action_B==7:
            self.agent_B_wood,collected=(self.collect_wood(self.agent_B_pos,self.agent_B_wood,"B"))

            if collected:
                reward+=4

        if action_A==8:
            if self.agent_A_wood>0:
                self.agent_A_wood-=1
                self.agent_A_energy=min(100,self.agent_A_energy+5)
                reward+=0.5


        if action_B==8:
            if self.agent_B_wood>0:
                self.agent_B_wood-=1
                self.agent_B_energy=min(100,self.agent_B_energy+30)
                reward+=1


        if action_A==9:
            if(self.agent_A_wood>0 and self.agent_B_food>0 and self.are_adjacent(self.agent_A_pos,self.agent_B_pos)):
                self.agent_A_wood-=1
                self.agent_B_wood+=1
                self.agent_B_food-=1
                self.agent_A_food+=1
                self.times_traded+=1
                reward+=7


        if action_B==9:

            if(self.agent_B_wood>0 and self.agent_A_food>0 and self.are_adjacent(self.agent_A_pos,self.agent_B_pos)):
                self.agent_B_wood-=1
                self.agent_A_wood+=1
                self.agent_A_food-=1
                self.agent_B_food+=1
                self.times_traded+=1
                reward+=7

        if both_at_goal:
            reward+=10

        terminated=(
            both_at_goal
            or
            self.agent_A_energy<=0
            or
            self.agent_B_energy<=0)

        truncated=(self.step_count>=self.max_steps)
        observation=self.observations()

        info={
            "times_closer":self.times_closer,
            "times_shared":self.times_shared,
            "times_traded":self.times_traded,
            "times_food_collected": self.times_food_collected,
            "times_wood_collected": self.times_wood_collected,
            "both_at_goal":both_at_goal

        }


        return (observation, reward, terminated, truncated,info)


In [ ]:
class QNetwork(nn.Module):

    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(60,128),
            nn.ReLU(),
            nn.Linear(128,128),
            nn.ReLU(),
            nn.Linear(128,10)
        )

    def forward(self, x):
        return self.network(x)

class ReplayBuffer:
    def __init__(self,capacity=10000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state,action,reward,next_state,done))

    def make_history(self, idx, length=4):
        history=[]

        for i in range(length):
             past_idx=idx-(length-1-i)
             if past_idx<0:
                 past_idx=0

             if self.buffer[past_idx][4]==True:
                 break

             history.append(self.buffer[past_idx][0])

        if len(history) == 0:
              history.append(self.buffer[idx][0])
        while len(history)<length:
              history.insert(0,history[0])


        return np.stack(history)

    def make_next_history(self, idx,history):

        next_history=list(history[1:].copy())
        next_history.append(self.buffer[idx][3])

        return np.stack(next_history)

    def sample(self, batch_size):
          indices=random.sample(range(len(self.buffer)),batch_size)

          states=[]
          actions=[]
          rewards=[]
          next_states=[]
          dones=[]

          for idx in indices:
                state_history = self.make_history(idx)
                next_state_history=self.make_next_history(idx,state_history)
                state, action, reward, next_state, done=self.buffer[idx]
                states.append(state_history.flatten())
                next_states.append(next_state_history.flatten())
                actions.append(action)
                rewards.append(reward)
                dones.append(done)

          return (
        np.array(states, dtype=np.float32),
        np.array(actions, dtype=np.int64),
        np.array(rewards, dtype=np.float32),
        np.array(next_states, dtype=np.float32),
        np.array(dones, dtype=np.float32)
    )


    def __len__(self):
        return len(self.buffer)


class IQLAgent:
    def __init__(self,learning_rate=1e-3,gamma=0.99,max_length=4):

        self.gamma=gamma
        self.q_network=QNetwork()
        self.target_network=QNetwork()
        self.history=deque(maxlen=max_length)
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.optimizer=optim.Adam(self.q_network.parameters(),lr=learning_rate)
        self.replay_buffer=ReplayBuffer()

    def reset_history(self,observation):
        self.history.clear()
        for _ in range(4):
            self.history.append(observation)

    def phi(self,observation,max_length=4):
      self.history.append(observation)
      return np.stack(self.history,axis=0).flatten()

    def select_action(self,observation,epsilon):
        history_state=self.phi(observation)
        if random.random()<epsilon:
            return random.randrange(6)


        history_state=torch.tensor(history_state,dtype=torch.float32).unsqueeze(0)

        with torch.no_grad():
            q_values=self.q_network(history_state)

        return q_values.argmax(dim=1).item()

    def update_target_network(self):
        self.target_network.load_state_dict(self.q_network.state_dict())

In [ ]:
num_episodes=2000
batch_size=64
epsilon_start=1.0
epsilon_end=0.05
epsilon_decay=0.995
target_update=100

env=MultiAgentGrid()
agents=[IQLAgent(),IQLAgent()]
epsilon=epsilon_start


for episode in range(num_episodes):
    observations,info=env.reset()
    episode_reward=0
    done=False

    while not done:
        actions=[]

        for i in range(2):
            action=agents[i].select_action(observations[i],epsilon)
            actions.append(action)

        next_observations, reward, terminated, truncated, info=env.step(actions)
        done=terminated or truncated


        for i in range(2):
            agents[i].replay_buffer.push(
                observations[i],
                actions[i],
                reward,
                next_observations[i],done)

        episode_reward+=reward

        for agent in agents:
            if len(agent.replay_buffer)>=batch_size:
              states,actions_b,rewards,next_states,dones=agent.replay_buffer.sample(batch_size)
              states=torch.tensor(states,dtype=torch.float32)
              actions_b=torch.tensor(actions_b).unsqueeze(1)
              rewards=torch.tensor(rewards,dtype=torch.float32)
              next_states=torch.tensor(next_states,dtype=torch.float32)

              dones = torch.tensor(
                    dones,
                    dtype=torch.float32
                )


              q_values=agent.q_network(states)
              current_q=q_values.gather(1,actions_b).squeeze(1)


              with torch.no_grad():
                    next_q_values=agent.target_network(next_states)
                    max_next_q=next_q_values.max(dim=1).values
                    target_q=(rewards+agent.gamma*max_next_q*(1-dones))

              loss=nn.functional.mse_loss(current_q,target_q)
              agent.optimizer.zero_grad()
              loss.backward()
              agent.optimizer.step()

        observations=next_observations

    epsilon=max(epsilon_end,epsilon*epsilon_decay)

    if episode % target_update==0:
        for agent in agents:
            agent.update_target_network()

    if episode % 10==0:
      print(f"episode {episode} episode reward={episode_reward} epsilon {epsilon:.3f}")
      print(info)


In [ ]:
class VDNReplayBuffer:

    def __init__(self, capacity=10000):
        self.buffer=deque(maxlen=capacity)


    def push(self,state_A,state_B,action_A,action_B,reward,next_state_A,next_state_B,done):
        self.buffer.append((state_A,state_B,action_A,action_B,reward,next_state_A,next_state_B,done))

    def make_history_A(self, idx, length=4):
        history=[]

        for i in range(length):
             past_idx=idx-(length-1-i)
             if past_idx<0:
                 past_idx=0

             if self.buffer[past_idx][7]==True:
                 break

             history.append(self.buffer[past_idx][0])

        if len(history) == 0:
              history.append(self.buffer[idx][0])
        while len(history)<length:
              history.insert(0,history[0])
        return np.stack(history)

    def make_history_B(self, idx, length=4):
        history=[]

        for i in range(length):
             past_idx=idx-(length-1-i)
             if past_idx<0:
                 past_idx=0

             if self.buffer[past_idx][7]==True:
                 break

             history.append(self.buffer[past_idx][1])

        if len(history) == 0:
              history.append(self.buffer[idx][1])
        while len(history)<length:
              history.insert(0,history[0])
        return np.stack(history)



    def make_next_history_A(self, idx, history):

        next_history=list(history[1:].copy())
        next_history.append(self.buffer[idx][5])

        return np.stack(next_history)

    def make_next_history_B(self, idx, history):

        next_history=list(history[1:].copy())
        next_history.append(self.buffer[idx][6])

        return np.stack(next_history)


    def sample(self, batch_size):
        indices=random.sample(range(len(self.buffer)),batch_size)

        states_A=[]
        states_B=[]
        actions_A=[]
        actions_B=[]
        rewards=[]
        next_states_A=[]
        next_states_B=[]
        dones=[]

        for idx in indices:
            state_history_A=self.make_history_A(idx)
            next_state_history_A=self.make_next_history_A(idx,state_history_A)
            state_history_B=self.make_history_B(idx)
            next_state_history_B=self.make_next_history_B(idx,state_history_B)
            state_A,state_B,action_A,action_B,reward,next_state_A,next_state_B,done=self.buffer[idx]
            states_A.append(state_history_A.flatten())
            states_B.append(state_history_B.flatten())
            next_states_A.append(next_state_history_A.flatten())
            next_states_B.append(next_state_history_B.flatten())
            actions_A.append(action_A)
            actions_B.append(action_B)
            rewards.append(reward)
            dones.append(done)

        return (
            np.array(states_A, dtype=np.float32),
            np.array(states_B, dtype=np.float32),
            np.array(actions_A, dtype=np.int64),
            np.array(actions_B, dtype=np.int64),
            np.array(rewards, dtype=np.float32),
            np.array(next_states_A, dtype=np.float32),
            np.array(next_states_B, dtype=np.float32),
            np.array(dones, dtype=np.float32)
        )

    def __len__(self):
        return len(self.buffer)



In [ ]:
class VDN:

    def __init__(self,learning_rate=1e-3,gamma=0.99,max_length=4):
        self.gamma=gamma
        self.q_A=QNetwork()
        self.history=deque(maxlen=max_length)
        self.target_A=QNetwork()
        self.target_A.load_state_dict(self.q_A.state_dict())
        self.q_B=QNetwork()
        self.target_B=QNetwork()
        self.target_B.load_state_dict(self.q_B.state_dict())
        self.optimizer=optim.Adam(
            list(self.q_A.parameters())+
            list(self.q_B.parameters()),lr=learning_rate)
        self.replay_buffer=VDNReplayBuffer()
        self.history_A=deque(maxlen=max_length)
        self.history_B=deque(maxlen=max_length)
        self.max_length=max_length


    def reset_histories(self,observation_A,observation_B):
        self.history_A.clear()
        self.history_B.clear()

        for _ in range(self.max_length):
            self.history_A.append(observation_A)
            self.history_B.append(observation_B)


    def phi_A(self,observation,max_length=4):
      self.history_A.append(observation)
      return np.stack(self.history_A,axis=0).flatten()


    def phi_B(self,observation,max_length=4):
      self.history_B.append(observation)
      return np.stack(self.history_B,axis=0).flatten()


    def select_action_A(self,observation,epsilon):
        history_state=self.phi_A(observation)
        if random.random() < epsilon:
           if random.random() < 0.2:
              return 9
           else:
              return random.randrange(9)

        history_state=torch.tensor(history_state,dtype=torch.float32).unsqueeze(0)

        with torch.no_grad():
            q_values=self.q_A(history_state)

        return q_values.argmax(dim=1).item()


    def select_action_B(self,observation,epsilon):
        history_state=self.phi_B(observation)
        if random.random()<epsilon:
          if random.random() < 0.2:
             return 9
          else:
             return random.randrange(9)


        history_state=torch.tensor(history_state,dtype=torch.float32).unsqueeze(0)

        with torch.no_grad():
            q_values=self.q_B(history_state)

        return q_values.argmax(dim=1).item()



    def update_target_network(self):
        self.target_A.load_state_dict(self.q_A.state_dict())
        self.target_B.load_state_dict(self.q_B.state_dict())

In [ ]:
vdn=VDN()
env=MultiAgentGrid()
num_epsiodes=2000
epsilon_start=1.0
batch_size=64
epsilon_end=0.05
epsilon_decay=0.995
target_update=100
epsilon=epsilon_start

for episode in range(num_epsiodes):
  observations,info=env.reset()
  done=False
  episode_reward=0.0

  vdn.reset_histories(observations[0],observations[1])


  while not done:
        action_A=vdn.select_action_A(observations[0],epsilon)
        action_B=vdn.select_action_B(observations[1],epsilon)
        actions=[action_A,action_B]

        next_observations, reward, terminated, truncated, info=env.step(actions)
        done=terminated or truncated




        vdn.replay_buffer.push(observations[0],observations[1],action_A,action_B,reward,next_observations[0],next_observations[1],done)

        if len(vdn.replay_buffer)>=batch_size:
              states_A,states_B,actions_A,actions_B,rewards,next_states_A,next_states_B,dones=vdn.replay_buffer.sample(batch_size)
              states_A=torch.tensor(states_A,dtype=torch.float32)
              states_B=torch.tensor(states_B,dtype=torch.float32)
              actions_A=torch.tensor(actions_A).unsqueeze(1)
              actions_B=torch.tensor(actions_B).unsqueeze(1)
              rewards=torch.tensor(rewards,dtype=torch.float32)
              next_states_A=torch.tensor(next_states_A,dtype=torch.float32)
              next_states_B=torch.tensor(next_states_B,dtype=torch.float32)
              dones=torch.tensor(dones,dtype=torch.float32)
              qnet_A=vdn.q_A(states_A)
              qnet_B=vdn.q_B(states_B)
              q_A_taken=qnet_A.gather(1,actions_A).squeeze(1)
              q_B_taken=qnet_B.gather(1,actions_B).squeeze(1)
              q_tot=(q_A_taken+q_B_taken)


              with torch.no_grad():
                next_q_A=vdn.target_A(next_states_A)
                next_q_B=vdn.target_B(next_states_B)
                max_next_q_A=next_q_A.max(dim=1).values
                max_next_q_B=next_q_B.max(dim=1).values
                next_q_tot=(max_next_q_A+max_next_q_B)
                target=(rewards+vdn.gamma*next_q_tot*(1-dones))




              loss=nn.functional.mse_loss(q_tot,target)
              vdn.optimizer.zero_grad()
              loss.backward()
              torch.nn.utils.clip_grad_norm_(list(vdn.q_A.parameters())+
                                                 list(vdn.q_B.parameters()), 10.0)
              vdn.optimizer.step()

        observations=next_observations
        episode_reward+=reward

  epsilon=max(epsilon_end,epsilon*epsilon_decay)


  if episode % target_update==0:
           vdn.update_target_network()

  if episode % 10==0:
           print(f"episode {episode} episode reward={episode_reward} epsilon {epsilon:.3f}")
           print(info)

In [ ]:
class DRQN(nn.Module):
    def __init__(self, obs_dim, action_dim, hidden_dim=64):
        super().__init__()
        self.obs_dim=obs_dim
        self.action_dim=action_dim
        self.hidden_dim=hidden_dim
        self.fc=nn.Linear(obs_dim+action_dim,hidden_dim)
        self.gru=nn.GRU(input_size=hidden_dim,hidden_size=hidden_dim,batch_first=True)
        self.q_head=nn.Linear(hidden_dim,action_dim)

    def forward(self, obs, prev_action, hidden=None):
        x=torch.cat([obs,prev_action],dim=-1)
        x=torch.relu(self.fc(x))
        x,hidden=self.gru(x,hidden)
        q_values=self.q_head(x)

        return q_values,hidden

In [ ]:
class QMIXReplayBuffer:

    def __init__(self, capacity=10000,sequence_length=4):

        self.buffer=deque(maxlen=capacity)
        self.sequence_length=sequence_length

    def push(self, obs_A, prev_action_A, action_A, obs_B, prev_action_B, action_B, reward, next_obs_A, next_prev_action_A, next_obs_B, next_prev_action_B, state, next_state, done):
        self.buffer.append((
            np.array(obs_A,dtype=np.float32),
            np.array(prev_action_A,dtype=np.float32),
            action_A,
            np.array(obs_B,dtype=np.float32),
            np.array(prev_action_B,dtype=np.float32),
            action_B,
            reward,
            np.array(next_obs_A,dtype=np.float32),
            np.array(next_prev_action_A,dtype=np.float32),
            np.array(next_obs_B,dtype=np.float32),
            np.array(next_prev_action_B,dtype=np.float32),
            np.array(state,dtype=np.float32),
            np.array(next_state,dtype=np.float32),
            done
        ))


    def make_history_A(self, idx):

        history=[]
        start=max(0,idx-self.sequence_length+1)

        for i in range(start,idx+1):
            if i>start:
                previous_done=self.buffer[i-1][13]
                if previous_done:
                    history=[]

            history.append(self.buffer[i][0])

        if len(history)==0:
            history.append(self.buffer[idx][0])

        while len(history)<self.sequence_length:
            history.insert(0,history[0])

        return np.stack(history)


    def make_history_B(self, idx):

        history=[]
        start=max(0,idx-self.sequence_length+1)

        for i in range(start,idx+1):
            if i>start:
                previous_done=self.buffer[i-1][13]
                if previous_done:
                    history=[]


            history.append(self.buffer[i][3])

        if len(history)==0:
            history.append(self.buffer[idx][3])

        while len(history)<self.sequence_length:
            history.insert(0,history[0])

        return np.stack(history)


    def make_prev_action_history_A(self, idx):

        history=[]
        start=max(0,idx-self.sequence_length+1)

        for i in range(start,idx+1):
            if i>start:
                previous_done=self.buffer[i-1][13]
                if previous_done:
                    history=[]


            history.append(self.buffer[i][1])


        if len(history)==0:
            history.append(self.buffer[idx][1])


        while len(history)<self.sequence_length:
            history.insert(0,history[0])

        return np.stack(history)


    def make_prev_action_history_B(self,idx):
        history=[]
        start=max(0,idx-self.sequence_length+1)

        for i in range(start,idx+1):
            if i>start:
                previous_done=self.buffer[i-1][13]
                if previous_done:
                    history=[]


            history.append(self.buffer[i][4])


        if len(history)==0:
            history.append(self.buffer[idx][4])


        while len(history)<self.sequence_length:
            history.insert(0,history[0])

        return np.stack(history)


    def make_next_history_A(self,idx):
        history=self.make_history_A(idx)
        next_history=list(history[1:])
        next_history.append(self.buffer[idx][7])
        return np.stack(next_history)


    def make_next_history_B(self, idx):
        history=self.make_history_B(idx)
        next_history=list(history[1:])
        next_history.append(self.buffer[idx][9])
        return np.stack(next_history)


    def make_next_prev_action_history_A(self, idx):
        history=self.make_prev_action_history_A(idx)
        next_history=list(history[1:])
        next_history.append(self.buffer[idx][8])
        return np.stack(next_history)


    def make_next_prev_action_history_B(self, idx):
        history=self.make_prev_action_history_B(idx)
        next_history=list(history[1:])
        next_history.append(self.buffer[idx][10])
        return np.stack(next_history)


    def sample(self, batch_size):

        indices=random.sample(range(len(self.buffer)),batch_size)
        states_A=[]
        prev_actions_A=[]
        states_B=[]
        prev_actions_B=[]
        actions_A=[]
        actions_B=[]
        rewards=[]
        next_states_A=[]
        next_prev_actions_A=[]
        next_states_B=[]
        next_prev_actions_B=[]
        states=[]
        next_states=[]
        dones=[]


        for idx in indices:
            history_A=self.make_history_A(idx)
            history_B=self.make_history_B(idx)
            prev_history_A=(self.make_prev_action_history_A(idx))
            prev_history_B=(self.make_prev_action_history_B(idx))
            next_history_A=(self.make_next_history_A(idx))
            next_history_B=(self.make_next_history_B(idx))
            next_prev_history_A=(self.make_next_prev_action_history_A(idx))
            next_prev_history_B=(self.make_next_prev_action_history_B(idx))

            transition=self.buffer[idx]
            action_A=transition[2]
            action_B=transition[5]
            reward=transition[6]
            state=transition[11]
            next_state=transition[12]
            done=transition[13]


            states_A.append(history_A)
            prev_actions_A.append(prev_history_A)
            states_B.append(history_B)
            prev_actions_B.append(prev_history_B)
            next_states_A.append(next_history_A)
            next_prev_actions_A.append(next_prev_history_A)
            next_states_B.append(next_history_B)
            next_prev_actions_B.append(next_prev_history_B)
            actions_A.append(action_A)
            actions_B.append(action_B)
            rewards.append(reward)
            states.append(state)
            next_states.append(next_state)
            dones.append(done)


        return(
            np.array(states_A,dtype=np.float32),
            np.array(prev_actions_A,dtype=np.float32),
            np.array(actions_A,dtype=np.int64),
            np.array(states_B,dtype=np.float32),
            np.array(prev_actions_B,dtype=np.float32),
            np.array(actions_B,dtype=np.int64),
            np.array(rewards,dtype=np.float32),
            np.array(next_states_A,dtype=np.float32),
            np.array(next_prev_actions_A,dtype=np.float32),
            np.array(next_states_B,dtype=np.float32),
            np.array(next_prev_actions_B,dtype=np.float32),
            np.array(states,dtype=np.float32),
            np.array(next_states,dtype=np.float32),
            np.array(dones,dtype=np.float32)
        )


    def __len__(self):
        return len(self.buffer)


In [ ]:
class QMixer(nn.Module):
    def __init__(self,agents,state_dim,mixing_dim=32):
        super().__init__()

        self.agents=agents
        self.state_dim=state_dim
        self.mixing_dim=mixing_dim
        self.hyper_w1=nn.Linear(state_dim,agents*mixing_dim)
        self.hyper_w2=nn.Linear(state_dim,mixing_dim)
        self.hyper_b1=nn.Linear(state_dim,mixing_dim)
        self.hyper_b2=nn.Sequential(nn.Linear(state_dim,mixing_dim),
                                    nn.ReLU(),
                                    nn.Linear(mixing_dim,1))


    def forward(self,agent_qs,state):

        w1=self.hyper_w1(state)
        w1=torch.abs(w1)
        w1=w1.view(-1,self.agents,self.mixing_dim)
        w2=self.hyper_w2(state)
        w2=torch.abs(w2)
        w2=w2.view(-1,self.mixing_dim,1)
        b1=self.hyper_b1(state)
        b1=b1.view(-1,1,self.mixing_dim)
        agent_qs=agent_qs.view(-1,1,self.agents)
        hidden=torch.bmm(agent_qs,w1)
        hidden=hidden+b1
        hidden=F.elu(hidden)
        b2=self.hyper_b2(state)
        q_tot=torch.bmm(hidden,w2)
        q_tot=q_tot+b2.view(-1,1,1)


        return q_tot.squeeze(-1)

In [ ]:
class QMIX:

    def __init__(
        self,
        obs_dim=15,
        action_dim=10,
        state_dim=30,
        hidden_dim=64,
        sequence_length=4,
        mixing_dim=32,
        learning_rate=1e-4,
        gamma=0.99
    ):

        self.gamma=gamma
        self.sequence_length=sequence_length
        self.action_dim=action_dim
        self.q_A=DRQN(obs_dim=obs_dim,
            action_dim=action_dim,
            hidden_dim=hidden_dim)

        self.q_B=DRQN(obs_dim=obs_dim,
            action_dim=action_dim,
            hidden_dim=hidden_dim)

        self.target_A=DRQN(
            obs_dim=obs_dim,
            action_dim=action_dim,
            hidden_dim=hidden_dim)

        self.target_B=DRQN(
            obs_dim=obs_dim,
            action_dim=action_dim,
            hidden_dim=hidden_dim)

        self.target_A.load_state_dict(self.q_A.state_dict())
        self.target_B.load_state_dict(self.q_B.state_dict())
        self.mixer=QMixer(agents=2,state_dim=state_dim,mixing_dim=mixing_dim)
        self.target_mixer=QMixer(agents=2,state_dim=state_dim,mixing_dim=mixing_dim)
        self.target_mixer.load_state_dict(self.mixer.state_dict())

        self.optimizer=optim.Adam(list(self.q_A.parameters())+list(self.q_B.parameters())+list(self.mixer.parameters()),lr=learning_rate)
        self.replay_buffer=QMIXReplayBuffer(sequence_length=sequence_length)


    def select_action_A(self,obs_sequence,prev_action_sequence,epsilon):

        if random.random()<epsilon:
            return random.randrange(self.action_dim)

        obs_tensor=torch.tensor(obs_sequence,dtype=torch.float32).unsqueeze(0)
        prev_action_tensor=torch.tensor(prev_action_sequence,dtype=torch.float32).unsqueeze(0)


        with torch.no_grad():
            q_values, _=self.q_A(obs_tensor,prev_action_tensor)

        q_values=q_values[:, -1, :]

        return q_values.argmax(dim=1).item()


    def select_action_B(self,obs_sequence,prev_action_sequence,epsilon):
        if random.random()<epsilon:
          return random.randrange(self.action_dim)

        obs_tensor=torch.tensor(obs_sequence,dtype=torch.float32).unsqueeze(0)
        prev_action_tensor=torch.tensor(prev_action_sequence,dtype=torch.float32).unsqueeze(0)

        with torch.no_grad():
            q_values,_=self.q_B(obs_tensor,prev_action_tensor)

        q_values=q_values[:, -1, :]

        return q_values.argmax(dim=1).item()


    def update_target_networks(self):
        self.target_A.load_state_dict(self.q_A.state_dict())
        self.target_B.load_state_dict(self.q_B.state_dict())
        self.target_mixer.load_state_dict(self.mixer.state_dict())


In [ ]:
import torch.nn.functional as F

In [ ]:
num_episodes=1500
obs_dim=15
action_dim=10
state_dim=30
sequence_length= 4
batch_size=32
gamma=0.99
learning_rate=1e-4
epsilon_start=1.0
epsilon_end=0.05
epsilon_decay=0.995
target_update=100
env=MultiAgentGrid()

qmix=QMIX(obs_dim=obs_dim,
    action_dim=action_dim,
    state_dim=state_dim,
    hidden_dim=64,
    sequence_length=sequence_length,
    mixing_dim=32,
    learning_rate=learning_rate,
    gamma=gamma
)

epsilon=epsilon_start
episode_rewards=[]
episode_shared=[]
episode_traded=[]
episode_closer=[]
episode_goal=[]
total_steps=0



for episode in range(num_episodes):

    observations, info=env.reset()
    obs_history_A=deque(maxlen=sequence_length)
    obs_history_B=deque(maxlen=sequence_length)
    prev_action_history_A=deque(maxlen=sequence_length)
    prev_action_history_B=deque(maxlen=sequence_length)

    for _ in range(sequence_length):
        obs_history_A.append(observations[0])
        obs_history_B.append(observations[1])
        prev_action_history_A.append(np.zeros(10,dtype=np.float32))
        prev_action_history_B.append(np.zeros(action_dim, dtype=np.float32))

    episode_reward=0
    done=False

    while not done:
        obs_seq_A=np.array(obs_history_A, dtype=np.float32)
        obs_seq_B=np.array(obs_history_B, dtype=np.float32)
        prev_action_seq_A=np.array(prev_action_history_A,dtype=np.float32)
        prev_action_seq_B=np.array(prev_action_history_B,dtype=np.float32)

        action_A=qmix.select_action_A(obs_seq_A,prev_action_seq_A,epsilon)
        action_B=qmix.select_action_B(obs_seq_B,prev_action_seq_B,epsilon)

        prev_action_A=prev_action_seq_A[-1].copy()
        prev_action_B=prev_action_seq_B[-1].copy()

        state=observations.flatten().astype(np.float32)

        next_observations, reward, terminated, truncated, info=env.step([action_A, action_B])
        done=terminated or truncated

        next_prev_action_A=np.zeros(action_dim,dtype=np.float32)
        next_prev_action_B=np.zeros(action_dim,dtype=np.float32)

        next_prev_action_A[action_A]=1.0
        next_prev_action_B[action_B]=1.0

        next_state=next_observations.flatten().astype(np.float32)

        qmix.replay_buffer.push(
            obs_A=observations[0],
            prev_action_A=prev_action_A,
            action_A=action_A,

            obs_B=observations[1],
            prev_action_B=prev_action_B,
            action_B=action_B,

            reward=reward,

            next_obs_A=next_observations[0],
            next_prev_action_A=next_prev_action_A,

            next_obs_B=next_observations[1],
            next_prev_action_B=next_prev_action_B,

            state=state,
            next_state=next_state,
            done=done
        )

        obs_history_A.append(next_observations[0])
        obs_history_B.append(next_observations[1])

        prev_action_history_A.append(next_prev_action_A)
        prev_action_history_B.append(next_prev_action_B)

        observations=next_observations
        episode_reward+=reward
        total_steps+=1

        if len(qmix.replay_buffer)>=batch_size:

            (obs_A_batch,prev_A_batch,actions_A_batch,obs_B_batch,prev_B_batch,actions_B_batch,
            rewards_batch,next_obs_A_batch,next_prev_A_batch,next_obs_B_batch,next_prev_B_batch,
            states_batch,next_states_batch,dones_batch)=qmix.replay_buffer.sample(batch_size)


            obs_A_batch=torch.tensor(obs_A_batch,dtype=torch.float32)
            prev_A_batch=torch.tensor(prev_A_batch,dtype=torch.float32)
            actions_A_batch=torch.tensor(actions_A_batch,dtype=torch.long)

            obs_B_batch=torch.tensor(obs_B_batch,dtype=torch.float32)
            prev_B_batch=torch.tensor(prev_B_batch,dtype=torch.float32)
            actions_B_batch=torch.tensor(actions_B_batch,dtype=torch.long)

            rewards_batch=torch.tensor(rewards_batch, dtype=torch.float32)

            next_obs_A_batch=torch.tensor(next_obs_A_batch,dtype=torch.float32)
            next_prev_A_batch=torch.tensor(next_prev_A_batch,dtype=torch.float32)
            next_obs_B_batch=torch.tensor(next_obs_B_batch,dtype=torch.float32)
            next_prev_B_batch=torch.tensor(next_prev_B_batch,dtype=torch.float32)

            states_batch=torch.tensor(states_batch,dtype=torch.float32)
            next_states_batch=torch.tensor(next_states_batch,dtype=torch.float32)
            dones_batch=torch.tensor(dones_batch,dtype=torch.float32)

            q_A_sequence,_=qmix.q_A(obs_A_batch,prev_A_batch)
            q_B_sequence,_=qmix.q_B(obs_B_batch,prev_B_batch)

            q_A=q_A_sequence[:, -1, :]
            q_B=q_B_sequence[:, -1, :]

            q_A_taken=q_A.gather(1,actions_A_batch.unsqueeze(1)).squeeze(1)
            q_B_taken=q_B.gather(1,actions_B_batch.unsqueeze(1)).squeeze(1)
            agent_qs=torch.stack([q_A_taken,q_B_taken],dim=1)
            q_total=qmix.mixer(agent_qs,states_batch)



            with torch.no_grad():

                target_q_A_sequence,_=qmix.target_A(next_obs_A_batch,next_prev_A_batch)
                target_q_B_sequence,_=qmix.target_B(next_obs_B_batch,next_prev_B_batch)

                target_q_A=target_q_A_sequence[:, -1, :]
                target_q_B=target_q_B_sequence[:, -1, :]

                max_q_A=target_q_A.max(dim=1).values
                max_q_B=target_q_B.max(dim=1).values
                target_agent_qs=torch.stack([max_q_A,max_q_B],dim=1)

                target_q_total=qmix.target_mixer(target_agent_qs,next_states_batch)
                target=rewards_batch+(gamma*target_q_total*(1-dones_batch))


            loss=F.mse_loss(q_total,target)
            qmix.optimizer.zero_grad()
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                list(qmix.q_A.parameters())+
                list(qmix.q_B.parameters())+
                list(qmix.mixer.parameters()),
                max_norm=10)

            qmix.optimizer.step()


        if total_steps%target_update==0:
            qmix.update_target_networks()


    epsilon=max(epsilon_end,epsilon*epsilon_decay)
    episode_rewards.append(episode_reward)
    episode_shared.append(info.get("times_shared",0))
    episode_traded.append(info.get("times_traded",0))
    episode_closer.append(info.get("times_closer", 0))
    episode_goal.append(info.get("both_at_goal", False))

    if episode % 10==0:
        print(f"episode {episode:4d}, reward:{episode_reward:.2f}, shared:{episode_shared[-1]:2d}, traded:{episode_traded[-1]:2d}, closer:{episode_closer[-1]:2d}, goal:{episode_goal[-1]}, epsilon: {epsilon:.3f}")

In [ ]:
torch.save({
    "q_A": agents[0].q_network.state_dict(),
    "q_B": agents[1].q_network.state_dict(),

    "target_A": agents[0].target_network.state_dict(),
    "target_B": agents[1].target_network.state_dict(),

    "optimizer_A": agents[0].optimizer.state_dict(),
    "optimizer_B": agents[1].optimizer.state_dict(),

    "epsilon": epsilon,

    "episode_rewards": episode_rewards,
    "episode_shared": episode_shared,
    "episode_traded": episode_traded,
    "episode_closer": episode_closer,
    "episode_goal": episode_goal,

}, "IQL_final.pth")

print("IQL saved!")

In [ ]:
torch.save({
    "q_A": vdn.q_A.state_dict(),
    "q_B": vdn.q_B.state_dict(),

    "target_A": vdn.target_A.state_dict(),
    "target_B": vdn.target_B.state_dict(),

    "optimizer": vdn.optimizer.state_dict(),

    "epsilon": epsilon,

    "episode_rewards": episode_rewards,
    "episode_shared": episode_shared,
    "episode_traded": episode_traded,
    "episode_closer": episode_closer,
    "episode_goal": episode_goal,

}, "VDN_final.pth")

print("VDN saved!")

In [ ]:
torch.save({
    "q_A": qmix.q_A.state_dict(),
    "q_B": qmix.q_B.state_dict(),

    "target_A": qmix.target_A.state_dict(),
    "target_B": qmix.target_B.state_dict(),

    "mixer": qmix.mixer.state_dict(),
    "target_mixer": qmix.target_mixer.state_dict(),

    "optimizer": qmix.optimizer.state_dict(),

    "epsilon": epsilon,

    "episode_rewards": episode_rewards,
    "episode_shared": episode_shared,
    "episode_traded": episode_traded,
    "episode_closer": episode_closer,
    "episode_goal": episode_goal,

}, "QMIX_final.pth")

In [ ]:
checkpoint = torch.load(
    "IQL_final.pth",
    map_location="cpu"
)

agents[0].q_network.load_state_dict(checkpoint["q_A"])
agents[1].q_network.load_state_dict(checkpoint["q_B"])

agents[0].target_network.load_state_dict(checkpoint["target_A"])
agents[1].target_network.load_state_dict(checkpoint["target_B"])

agents[0].optimizer.load_state_dict(checkpoint["optimizer_A"])
agents[1].optimizer.load_state_dict(checkpoint["optimizer_B"])

epsilon = checkpoint["epsilon"]

episode_rewards = checkpoint["episode_rewards"]
episode_shared = checkpoint["episode_shared"]
episode_traded = checkpoint["episode_traded"]
episode_closer = checkpoint["episode_closer"]
episode_goal = checkpoint["episode_goal"]

print("IQL loaded!")

In [ ]:
agents=[IQLAgent(),IQLAgent()]

In [ ]:
checkpoint = torch.load(
    "IQL_final.pth",
    map_location="cpu"
)

agents[0].q_network.load_state_dict(checkpoint["q_A"])
agents[1].q_network.load_state_dict(checkpoint["q_B"])

agents[0].target_network.load_state_dict(checkpoint["target_A"])
agents[1].target_network.load_state_dict(checkpoint["target_B"])

agents[0].optimizer.load_state_dict(checkpoint["optimizer_A"])
agents[1].optimizer.load_state_dict(checkpoint["optimizer_B"])

epsilon = checkpoint["epsilon"]

episode_rewards = checkpoint["episode_rewards"]
episode_shared = checkpoint["episode_shared"]
episode_traded = checkpoint["episode_traded"]
episode_closer = checkpoint["episode_closer"]
episode_goal = checkpoint["episode_goal"]

print("IQL loaded!")

In [ ]:
vdn = VDN(
    learning_rate=1e-3,
    gamma=0.99,
    max_length=4
)

checkpoint = torch.load(
    "VDN_final.pth",
    map_location="cpu"
)

vdn.q_A.load_state_dict(checkpoint["q_A"])
vdn.q_B.load_state_dict(checkpoint["q_B"])

vdn.target_A.load_state_dict(checkpoint["target_A"])
vdn.target_B.load_state_dict(checkpoint["target_B"])

vdn.optimizer.load_state_dict(checkpoint["optimizer"])

epsilon = checkpoint["epsilon"]

episode_rewards = checkpoint["episode_rewards"]
episode_shared = checkpoint["episode_shared"]
episode_traded = checkpoint["episode_traded"]
episode_closer = checkpoint["episode_closer"]
episode_goal = checkpoint["episode_goal"]

print("VDN loaded!")

In [ ]:
checkpoint = torch.load(
    "QMIX_final.pth",
    map_location="cpu"
)

qmix.q_A.load_state_dict(checkpoint["q_A"])
qmix.q_B.load_state_dict(checkpoint["q_B"])

qmix.target_A.load_state_dict(checkpoint["target_A"])
qmix.target_B.load_state_dict(checkpoint["target_B"])

qmix.mixer.load_state_dict(checkpoint["mixer"])
qmix.target_mixer.load_state_dict(checkpoint["target_mixer"])

qmix.optimizer.load_state_dict(checkpoint["optimizer"])

epsilon = checkpoint["epsilon"]

episode_rewards = checkpoint["episode_rewards"]
episode_shared = checkpoint["episode_shared"]
episode_traded = checkpoint["episode_traded"]
episode_closer = checkpoint["episode_closer"]
episode_goal = checkpoint["episode_goal"]

print("QMIX loaded!")

In [ ]:
from google.colab import files

files.download("IQL_final.pth")
files.download("VDN_final.pth")
files.download("QMIX_final.pth")

In [ ]:
qmix=QMIX(
    obs_dim=15,
    action_dim=10,
    state_dim=30,
    sequence_length=4
)

In [ ]:
checkpoint = torch.load(
    "QMIX_final.pth",
    map_location="cpu"
)

qmix.q_A.load_state_dict(checkpoint["q_A"])
qmix.q_B.load_state_dict(checkpoint["q_B"])

qmix.target_A.load_state_dict(checkpoint["target_A"])
qmix.target_B.load_state_dict(checkpoint["target_B"])

qmix.mixer.load_state_dict(checkpoint["mixer"])
qmix.target_mixer.load_state_dict(checkpoint["target_mixer"])

qmix.optimizer.load_state_dict(checkpoint["optimizer"])

epsilon = checkpoint["epsilon"]

episode_rewards = checkpoint["episode_rewards"]
episode_shared = checkpoint["episode_shared"]
episode_traded = checkpoint["episode_traded"]
episode_closer = checkpoint["episode_closer"]
episode_goal = checkpoint["episode_goal"]

print("QMIX loaded!")

In [ ]:
from google.colab import files
files.upload()

vdn=VDN(
    learning_rate=1e-3,
    gamma=0.99,
    max_length=4
)

checkpoint=torch.load(
    "VDN_final.pth",
    map_location="cpu"
)

vdn.q_A.load_state_dict(checkpoint["q_A"])
vdn.q_B.load_state_dict(checkpoint["q_B"])

vdn.target_A.load_state_dict(checkpoint["target_A"])
vdn.target_B.load_state_dict(checkpoint["target_B"])

vdn.optimizer.load_state_dict(checkpoint["optimizer"])

epsilon = checkpoint["epsilon"]

episode_rewards = checkpoint["episode_rewards"]
episode_shared = checkpoint["episode_shared"]
episode_traded = checkpoint["episode_traded"]
episode_closer = checkpoint["episode_closer"]
episode_goal = checkpoint["episode_goal"]

print("VDN loaded!")

In [ ]:
env = MultiAgentGrid()

In [ ]:
num_episodes=500
rewards=[]
shared=[]
traded=[]
closer=[]
goals=[]

epsilon=0.0

for episode in range(num_episodes):
  observations, info=env.reset()

  obs_history_A=deque(maxlen=4)
  obs_history_B=deque(maxlen=4)

  prev_action_history_A=deque(maxlen=4)
  prev_action_history_B=deque(maxlen=4)

  for _ in range(4):
    obs_history_A.append(observations[0])
    obs_history_B.append(observations[1])
    prev_action_history_A.append(np.zeros(10,dtype=np.float32))
    prev_action_history_B.append(np.zeros(10,dtype=np.float32))
    episode_reward=0
    done=False

    while not done:
      obs_seq_A=np.array(obs_history_A,dtype=np.float32)
      obs_seq_B=np.array(obs_history_B,dtype=np.float32)
      prev_seq_A=np.array(prev_action_history_A,dtype=np.float32)
      prev_seq_B=np.array(prev_action_history_B,dtype=np.float32)
      action_A=qmix.select_action_A(obs_seq_A,prev_seq_A,epsilon=0)
      action_B=qmix.select_action_B(obs_seq_B,prev_seq_B,epsilon=0)

      next_observations,reward,terminated,truncated,info=env.step([action_A, action_B])
      done=terminated or truncated

      next_prev_A=np.zeros(10,dtype=np.float32)
      next_prev_B=np.zeros(10,dtype=np.float32)

      next_prev_A[action_A]=1.0
      next_prev_B[action_B]=1.0

      obs_history_A.append(next_observations[0])
      obs_history_B.append(next_observations[1])

      prev_action_history_A.append(next_prev_A)
      prev_action_history_B.append(next_prev_B)

      observations=next_observations
      episode_reward+=reward

    rewards.append(episode_reward)
    shared.append(info.get("times_shared",0))
    traded.append(info.get("times_traded",0))
    closer.append(info.get("times_closer",0))
    goals.append(info.get("both_at_goal", False))



results={
        "average_reward":np.mean(rewards),
        "reward_std":np.std(rewards),
        "average_shared":np.mean(shared),
        "average_traded":np.mean(traded),
        "average_closer":np.mean(closer),
        "goal_rate":np.mean(goals)
    }

print(results)

In [ ]:
num_episodes=500

rewards=[]
shared=[]
traded=[]
closer=[]
goals=[]

for episode in range(num_episodes):
        observations, info=env.reset()
        vdn.reset_histories(observations[0],observations[1])
        done=False
        episode_reward=0

        while not done:
            action_A=vdn.select_action_A(observations[0],epsilon=0)
            action_B=vdn.select_action_B(observations[1],epsilon=0)

            next_observations,reward,terminated,truncated,info=env.step([action_A,action_B])
            done=terminated or truncated
            episode_reward+=reward
            observations=next_observations

        rewards.append(episode_reward)
        shared.append(info.get("times_shared", 0))
        traded.append(info.get("times_traded", 0))
        closer.append(info.get("times_closer", 0))
        goals.append(info.get("both_at_goal", False))

print({
        "average_reward":np.mean(rewards),
        "reward_std":np.std(rewards),
        "average_shared":np.mean(shared),
        "average_traded":np.mean(traded),
        "average_closer":np.mean(closer),
        "goal_rate":np.mean(goals)}
    )

In [ ]:
num_episodes=500
rewards=[]
shared=[]
traded=[]
closer=[]
goals=[]

for episode in range(num_episodes):

        observations, info=env.reset()
        episode_reward=0
        done=False
        agents[0].reset_history(observations[0])
        agents[1].reset_history(observations[1])

        while not done:
            actions=[]

            for i in range(2):
                action=agents[i].select_action(observations[i],epsilon=0)
                actions.append(action)

            next_observations,reward,terminated,truncated,info=env.step(actions)
            done=terminated or truncated
            episode_reward+=reward
            observations=next_observations


        rewards.append(episode_reward)
        shared.append(info.get("times_shared",0))
        traded.append(info.get("times_traded",0))
        closer.append(info.get("times_closer",0))
        goals.append(info.get("both_at_goal", False))


results={
        "average_reward": np.mean(rewards),
        "reward_std": np.std(rewards),
        "average_shared": np.mean(shared),
        "average_traded": np.mean(traded),
        "average_closer": np.mean(closer),
        "goal_rate": np.mean(goals)
    }

print(results)

In [ ]:
class PopulationAgent:

    def __init__(self,vdn_network,agent_id,history_length=4):
        self.agent_id=agent_id
        self.q_network=copy.deepcopy(vdn_network)
        self.history_length=history_length
        self.history=deque(maxlen=history_length)
        self.fitness=0.0
        self.total_reward=0.0
        self.total_shared=0
        self.total_traded=0
        self.total_closer=0
        self.total_steps=0
        self.q_network.eval()

    def reset_history(self, observation):
        self.history.clear()

        for _ in range(self.history_length):
            self.history.append(observation)

    def select_action(self, observation):
        self.history.append(observation)
        state=np.stack(self.history,axis=0).flatten()
        state=torch.tensor(state,dtype=torch.float32).unsqueeze(0)

        with torch.no_grad():
            q_values=self.q_network(state)

        return q_values.argmax(dim=1).item()

    def reset_statistics(self):
        self.fitness=0.0
        self.total_reward=0.0
        self.total_shared=0
        self.total_traded=0
        self.total_closer=0
        self.total_steps=0

In [ ]:
def mutate_network(network, mutation_strength=0.01):

    with torch.no_grad():
        for param in network.parameters():
            noise=torch.randn_like(param)*mutation_strength
            param.add_(noise)

In [ ]:
def create_next_generation(parents,population_size=20,mutation_strength=0.01):
    new_population=[]

    for agent_id in range(population_size):
        parent=random.choice(parents)
        child=PopulationAgent(vdn_network=parent.q_network,agent_id=agent_id,history_length=parent.history_length)

        mutate_network(child.q_network,mutation_strength=mutation_strength)

        new_population.append(child)

    return new_population

In [ ]:
def create_generation_zero(vdn_network,population_size=20,mutation_strength=0.01):
    population=[]

    for agent_id in range(population_size):
        agent=PopulationAgent(vdn_network=vdn_network,agent_id=agent_id,history_length=4)
        mutate_network(agent.q_network,mutation_strength)
        population.append(agent)

    return population

In [ ]:
generation_0=create_generation_zero(vdn.q_A,population_size=20,mutation_strength=0.01)

print("Generation 0 created:", len(generation_0))

In [ ]:
def calculate_fitness(agent, reward, shared, traded, closer):
    fitness=reward
    fitness+=2.0*shared
    fitness+=3.0*traded
    fitness+=0.1*closer
    return fitness

In [ ]:
def evaluate_population(population,env,episodes_per_pair=5):
    for agent in population:
        agent.reset_statistics()

    for i in range(len(population)):
        for j in range(i+1,len(population)):
            agent_A=population[i]
            agent_B=population[j]

            for episode in range(episodes_per_pair):
                observations,info=env.reset()
                agent_A.reset_history(observations[0])
                agent_B.reset_history(observations[1])
                done=False
                episode_reward=0

                while not done:
                    action_A=agent_A.select_action(observations[0])
                    action_B=agent_B.select_action(observations[1])
                    actions=[action_A,action_B]
                    next_observations,reward,terminated,truncated,info=env.step(actions)
                    done=terminated or truncated
                    episode_reward+=reward
                    observations=next_observations

                shared=info.get("times_shared",0)
                traded=info.get("times_traded",0)
                closer=info.get("times_closer",0)
                fitness_A=calculate_fitness(agent_A,episode_reward,shared,traded,closer)
                fitness_B=calculate_fitness(agent_B,episode_reward,shared,traded,closer)


                agent_A.fitness+=fitness_A
                agent_B.fitness+=fitness_B
                agent_A.total_reward+=episode_reward
                agent_B.total_reward+=episode_reward
                agent_A.total_shared+=shared
                agent_B.total_shared+=shared
                agent_A.total_traded+=traded
                agent_B.total_traded+=traded
                agent_A.total_closer+=closer
                agent_B.total_closer+=closer
                agent_A.total_steps+=env.step_count
                agent_B.total_steps+=env.step_count

    return population

In [ ]:
generation_0=evaluate_population(generation_0,env,episodes_per_pair=5)

In [ ]:
for agent in generation_0:
    print(f"ID:{agent.agent_id},fitness:{agent.fitness:.2f},reward:{agent.total_reward:.2f},shared:{agent.total_shared},traded:{agent.total_traded}")


In [ ]:
def select_parents(population,survival_rate):
    sorted_population=sorted(
        population,
        key=lambda agent:agent.fitness,
        reverse=True)

    return sorted_population[:2]

In [ ]:
parents_0=select_parents(generation_0,survival_rate=0.5)

for agent in parents_0:
    print(f"ID:{agent.agent_id}, fitness:{agent.fitness:.2f}, shared:{agent.total_shared}, traded:{agent.total_traded}")

In [ ]:
def create_next_generation(parents,population_size=20,mutation_strength=0.01):
    new_population=[]

    for agent_id in range(population_size):
        parent=random.choice(parents)
        child=PopulationAgent(
            vdn_network=parent.q_network,
            agent_id=agent_id,
            history_length=parent.history_length
        )

        mutate_network(
            child.q_network,
            mutation_strength=mutation_strength)

        new_population.append(child)

    return new_population

In [ ]:
generation_1=create_next_generation(parents_0,population_size=20,mutation_strength=0.01)

In [ ]:
generation_1=evaluate_population(generation_1,env,episodes_per_pair=5)

In [ ]:
for agent in generation_1:
    print(f"ID:{agent.agent_id},fitness:{agent.fitness:.2f},reward:{agent.total_reward:.2f},shared:{agent.total_shared},traded:{agent.total_traded}")

In [ ]:
parents_1=select_parents(generation_1,survival_rate=0.5)
for agent in parents_1:
    print(f"ID:{agent.agent_id}, fitness:{agent.fitness:.2f}, shared:{agent.total_shared}, traded:{agent.total_traded}")

In [ ]:
generation_2=create_next_generation(parents_1,population_size=20,mutation_strength=0.01)

In [ ]:
generation_2=evaluate_population(generation_2,env,episodes_per_pair=5)

In [ ]:
for agent in generation_2:
    print(f"ID:{agent.agent_id},fitness:{agent.fitness:.2f},reward:{agent.total_reward:.2f},shared:{agent.total_shared},traded:{agent.total_traded}")

In [ ]:
parents_2=select_parents(generation_2,survival_rate=0.5)

for agent in parents_2:
    print(f"ID:{agent.agent_id}, fitness:{agent.fitness:.2f}, shared:{agent.total_shared}, traded:{agent.total_traded}")

In [ ]:
generation_3=create_next_generation(
    parents_2,
    population_size=20,
    mutation_strength=0.01)

In [ ]:
generation_3=evaluate_population(generation_3,env,episodes_per_pair=5)

In [ ]:
for agent in generation_3:
    print(f"ID:{agent.agent_id},fitness:{agent.fitness:.2f},reward:{agent.total_reward:.2f},shared:{agent.total_shared},traded:{agent.total_traded}")